# Feature Engineering

## Objective

The objective of this notebook is to prepare the SWaT dataset for machine learning.

Main tasks:

- Encode target labels
- Prepare numerical features
- Handle missing values
- Scale numerical variables
- Build the feature matrix

In [1]:
import numpy as np
import pandas as pd


from pathlib import Path

from sklearn.preprocessing import StandardScaler

In [2]:
PROJECT_ROOT = Path.cwd().parent

DATA_PATH = PROJECT_ROOT/"data"/"processed"/"swat_clean_v1.csv"

df = pd.read_csv(DATA_PATH)

df["Timestamp"] = pd.to_datetime(df["Timestamp"])

df.head()

,Timestamp,FIT101,LIT101,MV101,P101,P102,AIT201,AIT202,AIT203,FIT201,...,P501,P502,PIT501,PIT502,PIT503,FIT601,P601,P602,P603,Normal/Attack
0,2015-12-28 10:00:00,2.427057,522.8467,2.0,2,1,262.0161,8.396437,328.6337,2.445391,...,2,1,250.8652,1.649953,189.5988,0.000128,1,1,1,Normal
1,2015-12-28 10:00:01,2.446274,522.8860,2.0,2,1,262.0161,8.396437,328.6337,2.445391,...,2,1,250.8652,1.649953,189.6789,0.000128,1,1,1,Normal
2,2015-12-28 10:00:02,2.489191,522.8467,2.0,2,1,262.0161,8.394514,328.6337,2.442316,...,2,1,250.8812,1.649953,189.6789,0.000128,1,1,1,Normal
3,2015-12-28 10:00:03,2.534350,522.9645,2.0,2,1,262.0161,8.394514,328.6337,2.442316,...,2,1,250.8812,1.649953,189.6148,0.000128,1,1,1,Normal
4,2015-12-28 10:00:04,2.569260,523.4748,2.0,2,1,262.0161,8.394514,328.6337,2.443085,...,2,1,250.8812,1.649953,189.5027,0.000128,1,1,1,Normal


## Target Encoding

Machine learning algorithms require numerical labels.

Normal → 0

Attack → 1

In [3]:
df["Target"] = (
    df["Normal/Attack"]
      .map({
          "Normal":0,
          "Attack":1
      })
)

df[["Normal/Attack","Target"]].head()

,Normal/Attack,Target
0,Normal,0
1,Normal,0
2,Normal,0
3,Normal,0
4,Normal,0


In [4]:
df["Target"].value_counts()

Target
0    1387098
1      54621
Name: count, dtype: int64

## Removing Non-Predictive Columns

Timestamp is retained separately for visualization.

The original target label is removed after encoding.

In [5]:
df_model = df.drop(
    columns=[
        "Timestamp",
        "Normal/Attack"
    ]
)

In [6]:
df_model.head()

,FIT101,LIT101,MV101,P101,P102,AIT201,AIT202,AIT203,FIT201,MV201,...,P501,P502,PIT501,PIT502,PIT503,FIT601,P601,P602,P603,Target
0,2.427057,522.8467,2.0,2,1,262.0161,8.396437,328.6337,2.445391,2.0,...,2,1,250.8652,1.649953,189.5988,0.000128,1,1,1,0
1,2.446274,522.8860,2.0,2,1,262.0161,8.396437,328.6337,2.445391,2.0,...,2,1,250.8652,1.649953,189.6789,0.000128,1,1,1,0
2,2.489191,522.8467,2.0,2,1,262.0161,8.394514,328.6337,2.442316,2.0,...,2,1,250.8812,1.649953,189.6789,0.000128,1,1,1,0
3,2.534350,522.9645,2.0,2,1,262.0161,8.394514,328.6337,2.442316,2.0,...,2,1,250.8812,1.649953,189.6148,0.000128,1,1,1,0
4,2.569260,523.4748,2.0,2,1,262.0161,8.394514,328.6337,2.443085,2.0,...,2,1,250.8812,1.649953,189.5027,0.000128,1,1,1,0


In [7]:
numeric_features = df_model.select_dtypes(include=np.number).columns.tolist()

print("Number of Features:",len(numeric_features))

Number of Features: 52


In [8]:
numeric_features[:10]

['FIT101',
 'LIT101',
 'MV101',
 'P101',
 'P102',
 'AIT201',
 'AIT202',
 'AIT203',
 'FIT201',
 'MV201']

## Missing Value Handling

Missing values are replaced using the median.

Median is preferred because it is robust against extreme values commonly found in industrial datasets.

In [9]:
missing_before = df_model.isnull().sum().sum()

print("Missing Before:",missing_before)

Missing Before: 6942600


In [10]:
df_model = df_model.fillna(df_model.median(numeric_only=True))

In [11]:
missing_after = df_model.isnull().sum().sum()

print("Missing After:",missing_after)

Missing After: 0


## Feature Scaling

Standardization transforms features to zero mean and unit variance.

This improves the performance of distance-based and gradient-based machine learning algorithms.

In [12]:
scaler = StandardScaler()

In [13]:
X = df_model.drop(columns="Target")

y = df_model["Target"]

In [14]:
X_scaled = scaler.fit_transform(X)

In [15]:
X_scaled = pd.DataFrame(
    X_scaled,
    columns=X.columns
)

X_scaled.head()

,FIT101,LIT101,MV101,P101,P102,AIT201,AIT202,AIT203,FIT201,MV201,...,FIT504,P501,P502,PIT501,PIT502,PIT503,FIT601,P601,P602,P603
0,0.538116,-0.567030,0.335953,0.60675,-0.046608,2.994988,-0.299856,-0.248463,0.603097,0.321655,...,0.165254,0.165048,0.0,0.143557,1.727161,0.133610,-0.098181,0.0,-0.091453,0.0
1,0.554769,-0.566712,0.335953,0.60675,-0.046608,2.994988,-0.299856,-0.248463,0.603097,0.321655,...,0.165254,0.165048,0.0,0.143557,1.727161,0.136315,-0.098181,0.0,-0.091453,0.0
2,0.591961,-0.567030,0.335953,0.60675,-0.046608,2.994988,-0.316133,-0.248463,0.600262,0.321655,...,0.181968,0.165048,0.0,0.143974,1.727161,0.136315,-0.098181,0.0,-0.091453,0.0
3,0.631096,-0.566079,0.335953,0.60675,-0.046608,2.994988,-0.316133,-0.248463,0.600262,0.321655,...,0.181968,0.165048,0.0,0.143974,1.727161,0.134150,-0.098181,0.0,-0.091453,0.0
4,0.661349,-0.561962,0.335953,0.60675,-0.046608,2.994988,-0.316133,-0.248463,0.600971,0.321655,...,0.181968,0.165048,0.0,0.143974,1.727161,0.130365,-0.098181,0.0,-0.091453,0.0


In [16]:
final_dataset = X_scaled.copy()

final_dataset["Target"] = y.values

final_dataset.head()

,FIT101,LIT101,MV101,P101,P102,AIT201,AIT202,AIT203,FIT201,MV201,...,P501,P502,PIT501,PIT502,PIT503,FIT601,P601,P602,P603,Target
0,0.538116,-0.567030,0.335953,0.60675,-0.046608,2.994988,-0.299856,-0.248463,0.603097,0.321655,...,0.165048,0.0,0.143557,1.727161,0.133610,-0.098181,0.0,-0.091453,0.0,0
1,0.554769,-0.566712,0.335953,0.60675,-0.046608,2.994988,-0.299856,-0.248463,0.603097,0.321655,...,0.165048,0.0,0.143557,1.727161,0.136315,-0.098181,0.0,-0.091453,0.0,0
2,0.591961,-0.567030,0.335953,0.60675,-0.046608,2.994988,-0.316133,-0.248463,0.600262,0.321655,...,0.165048,0.0,0.143974,1.727161,0.136315,-0.098181,0.0,-0.091453,0.0,0
3,0.631096,-0.566079,0.335953,0.60675,-0.046608,2.994988,-0.316133,-0.248463,0.600262,0.321655,...,0.165048,0.0,0.143974,1.727161,0.134150,-0.098181,0.0,-0.091453,0.0,0
4,0.661349,-0.561962,0.335953,0.60675,-0.046608,2.994988,-0.316133,-0.248463,0.600971,0.321655,...,0.165048,0.0,0.143974,1.727161,0.130365,-0.098181,0.0,-0.091453,0.0,0


In [ ]:
SAVE_PATH = PROJECT_ROOT/"data"/"processed"/"swat_features.csv"

final_dataset.to_csv(
    SAVE_PATH,
    index=False
)

print("Dataset Saved Successfully")

# Summary

Feature Engineering completed successfully.

Completed tasks:

✔ Target Encoding

✔ Missing Value Handling

✔ Feature Scaling

✔ Feature Matrix Preparation

The processed dataset is now ready for Feature Selection and Machine Learning.